# CSV Schema Profiler & Column Matcher
Reusable Jupyter notebook for **multiple CSV sources**. It identifies likely semantic types (phone, ZIP, state, address, person name, email, date, etc.), detects missing/extra columns, and ranks suggested mappings using **column names, inferred types, and sampled values**.

**Quick start:** Put CSV files in a `data/` folder beside this notebook, run all cells, and check `results/`. You can change the file selection and thresholds in **Configuration**. Results are *suggestions*; inspect sample values before mapping columns.

**Install once if necessary:** `%pip install pandas rapidfuzz` (RapidFuzz is optional; stdlib fallback is included).

In [ ]:
from pathlib import Path
import re, itertools, math, json, unicodedata
from difflib import SequenceMatcher
from collections import Counter
import pandas as pd
try:
    from rapidfuzz import fuzz
    def similarity(a,b): return fuzz.ratio(a,b) / 100
except ImportError:
    def similarity(a,b): return SequenceMatcher(None,a,b).ratio()

# ---- Configuration: edit these ----
DATA_DIR = Path('data')
OUT_DIR = Path('results')
SAMPLE_ROWS = 15000           # CSV rows read per file; increase for deeper profiles
PROFILE_VALUES = 750          # nonblank examples evaluated per column
TOP_MATCHES = 5               # candidate suggestions per column per comparison
MATCH_THRESHOLD = 0.49        # lower for broad candidates, raise for conservative mapping
USE_ALL_PAIRS = True          # True: compare every pair; False: choose pairs below
SELECTED_PAIRS = []           # e.g. [('old_export','new_export')]; filename stems
RANDOM_SEED = 42
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)
print('Place your .csv files into:', DATA_DIR.resolve())

## 1. Read data safely
All CSV fields load as **strings** to preserve leading zeros (ZIPs, IDs, phone numbers). The notebook samples up to `SAMPLE_ROWS` rows from each file and keeps original column names. To profile an entire file set `SAMPLE_ROWS = None`.

In [ ]:
def load_csv(path):
    kwargs = dict(dtype='string', keep_default_na=False, encoding='utf-8-sig', low_memory=False)
    if SAMPLE_ROWS is not None: kwargs['nrows'] = SAMPLE_ROWS
    try:
        frame = pd.read_csv(path, **kwargs)
    except UnicodeDecodeError:
        kwargs['encoding'] = 'cp1252'
        frame = pd.read_csv(path, **kwargs)
    frame.columns = [str(x) for x in frame.columns]
    return frame

paths = sorted(DATA_DIR.glob('*.csv'))
if not paths:
    print('No CSVs yet. Add files to data/ and rerun this cell.')
frames = {}
for path in paths:
    key = path.stem
    if key in frames: raise ValueError(f'Duplicate dataset name: {key}')
    frames[key] = load_csv(path)
print('Loaded:', {k: f'{len(v):,} rows × {len(v.columns)} cols' for k,v in frames.items()})

## 2. Normalize names & detect likely types
Type labels are heuristic, particularly for **person name vs. arbitrary text**, **IDs vs. numeric strings**, and **addresses**. The score is the proportion of sampled nonblank values matching a pattern (or a name-based hint where appropriate); it is **not a calibrated probability**.

In [ ]:
SYNONYMS = {
 'num':'number','nbr':'number','no':'number','tel':'phone','telephone':'phone',
 'mobile':'phone','cell':'phone','ph':'phone','postal':'zip','postcode':'zip',
 'zipcode':'zip','zip4':'zip','addr':'address','street':'address','st':'street',
 'fname':'first','lname':'last','surname':'last','given':'first',
 'cust':'customer','client':'customer','acct':'account','dt':'date',
 'dob':'birth date','statecode':'state','province':'state', 'e-mail':'email',
 'id':'id'
}
STATES = set('AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC'.split())
STATE_NAMES = set('alabama alaska arizona arkansas california colorado connecticut delaware florida georgia hawaii idaho illinois indiana iowa kansas kentucky louisiana maine maryland massachusetts michigan minnesota mississippi missouri montana nebraska nevada new hampshire new jersey new mexico new york north carolina north dakota ohio oklahoma oregon pennsylvania rhode island south carolina south dakota tennessee texas utah vermont virginia washington west virginia wisconsin district of columbia'.split('|')) if False else {
 'alabama','alaska','arizona','arkansas','california','colorado','connecticut','delaware','florida','georgia','hawaii','idaho','illinois','indiana','iowa','kansas','kentucky','louisiana','maine','maryland','massachusetts','michigan','minnesota','mississippi','missouri','montana','nebraska','nevada','new hampshire','new jersey','new mexico','new york','north carolina','north dakota','ohio','oklahoma','oregon','pennsylvania','rhode island','south carolina','south dakota','tennessee','texas','utah','vermont','virginia','washington','west virginia','wisconsin','district of columbia'}
KNOWN_HINTS = ['email','phone','zip','postal','state','address','street','city','first','last','name','date','time','amount','price','id','account','customer','order','country','latitude','longitude']
def tokenize_name(name):
    s = re.sub(r'([a-z0-9])([A-Z])',r'\1 \2',str(name))
    s = re.sub(r'([A-Z])([A-Z][a-z])',r'\1 \2',s)
    s = re.sub(r'[^a-zA-Z0-9]+',' ',s).lower().strip()
    tokens=[]
    for t in s.split():
        # collapse common compound names, e.g. PhoneNum, ZipCode
        if t in SYNONYMS: t = SYNONYMS[t]
        tokens.extend(t.split())
    return tokens

def normalized_name(s):
    return ' '.join(tokenize_name(s))

def is_blank(s):
    return str(s).strip().lower() in {'','null','none','nan','n/a','na','<na>'}

def classify_column(column, series):
    cleaned = series.dropna().astype(str).str.strip()
    cleaned = cleaned[~cleaned.map(is_blank)]
    sample = cleaned.sample(min(len(cleaned),PROFILE_VALUES),random_state=RANDOM_SEED) if len(cleaned) else cleaned
    vals = sample.tolist(); total=len(vals)
    name=normalized_name(column); token_set=set(name.split())
    if not total: return ('empty',0.0, {}, [], 0,0,0)
    checks = {
      'email': lambda s: bool(re.fullmatch(r'[^\s@]+@[^\s@]+\.[^\s@]+',s)),
      'phone': lambda s: bool(re.fullmatch(r'\+?[\d\s().-]+(?:\s*(?:ext|x)\s*\d+)?',s,re.I)) and 10<=len(re.sub(r'\D','',s))<=15,
      'zip': lambda s: bool(re.fullmatch(r'\d{5}(?:-\d{4})?',s)),
      'state': lambda s: s.upper() in STATES or s.casefold() in STATE_NAMES,
      'address': lambda s: bool(re.search(r'^\s*\d+[\w\s.,#/-]+\b(st|street|ave|avenue|rd|road|dr|drive|blvd|boulevard|lane|ln|way|court|ct|place|pl|pkwy|parkway)\b',s,re.I)),
      'integer': lambda s: bool(re.fullmatch(r'[+-]?\d+',s.replace(',',''))),
      'decimal': lambda s: bool(re.fullmatch(r'[+-]?(?:\d+[,.]?)*\d+(?:\.\d+)?',s.replace(',',''))),
      'boolean': lambda s: s.casefold() in {'true','false','yes','no','y','n','0','1'},
      'date': lambda s: bool(re.fullmatch(r'\d{4}-\d{1,2}-\d{1,2}(?:[tT ].*)?|\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}(?:\s+.*)?',s)),
      'url': lambda s: bool(re.match(r'https?://\S+',s,re.I)),
      'person_name': lambda s: bool(re.fullmatch(r"[A-Za-z][A-Za-z'\-]+(?:\s+[A-Za-z][A-Za-z'\-]+){1,3}",s)),
    }
    scores={k:sum(bool(fn(v)) for v in vals)/total for k,fn in checks.items()}
    hints = {
     'email': {'email'}, 'phone': {'phone'}, 'zip': {'zip'}, 'state': {'state'},
     'address': {'address','street'}, 'person_name': {'name','first','last'},
     'date': {'date','birth','created','updated','timestamp'},
     'integer': {'count','quantity'}, 'decimal': {'amount','price','cost','total','rate'},
     'url': {'url','website','link'}
    }
    # Prioritize specific semantic types over generic numeric/text.
    types=['email','url','phone','zip','state','address','date','boolean','integer','decimal','person_name']
    best='text'; confidence=1.0
    for typ in types:
        ratio=scores[typ]
        hinted=bool(token_set & hints.get(typ,set()))
        cutoff=0.70 if hinted else (0.85 if typ!='person_name' else 0.90)
        if typ=='person_name' and not hinted: continue  # prevents common prose from becoming names
        if typ=='date' and not hinted and scores['integer']>0.95: continue
        if ratio>=cutoff:
            best=typ; confidence=round(ratio,3); break
    # Name-bearing columns can be single words; mark conservatively
    if best=='text' and token_set & {'first','last','name'} and scores['person_name']>=0.30:
        best='person_name'; confidence=round(scores['person_name'],3)
    # Identifier labels outrank numeric pattern matches (e.g. 5-digit order IDs are not ZIP codes).
    if 'id' in token_set or 'identifier' in token_set or ('number' in token_set and not (token_set & {'phone','zip','account'})):
        if best in {'zip','phone','integer','decimal','text'}:
            best='identifier'; confidence=0.75
    distinct=cleaned.nunique()
    examples=cleaned.drop_duplicates().head(4).tolist()
    return best,confidence,{k:round(v,3) for k,v in scores.items()},examples,len(cleaned),distinct,len(series)

profiles=[]; meta={}
for dataset,df in frames.items():
    for pos,col in enumerate(df.columns):
        t,conf,scores,examples,pop,distinct,rows=classify_column(col,df.iloc[:,pos])
        key=(dataset,pos)
        meta[key]={'column':col,'normalized':normalized_name(col),'type':t,'confidence':conf,'scores':scores,
                   'examples':examples, 'position':pos}
        profiles.append({'Dataset':dataset,'Column':col,'Position':pos,'Normalized Name':normalized_name(col),
          'Suspected Type':t,'Type Score':conf,'Nonblank Rows':pop,'Total Rows':rows,
          'Fill %':round(100*pop/max(rows,1),1),'Distinct Values':distinct,
          'Distinct %':round(100*distinct/max(pop,1),1),
          'Examples':' | '.join(examples),'Pattern Scores':json.dumps(scores)})
profiles_df=pd.DataFrame(profiles)
display(profiles_df.drop(columns=['Pattern Scores']).head(30)) if len(profiles_df) else print('Nothing to profile yet.')

## 3. Compare columns across sources
The ranking includes: token-aware normalized **name similarity**, **type compatibility**, and **sampled value overlap**. Overlap can be useful for fields that share actual values, but unrelated ID fields may also overlap; scores must be reviewed manually. Do not assume the top result is a confirmed mapping.

`Matching` is intentionally **not** one-to-one: a column may have several plausible alternatives. `Best Matches` adds a single top candidate for quick review, with a confidence label. 

In [ ]:
TYPE_FAMILIES = [ {'integer','decimal','identifier'}, {'text','person_name'}, {'zip','integer','identifier'}, {'date'} ]
def type_compat(a,b):
    if a==b: return 1.0
    if a=='empty' or b=='empty': return 0.3
    if a=='text' or b=='text': return 0.58
    if any({a,b}<=family for family in TYPE_FAMILIES):return 0.76
    return 0.05

def name_sim(a,b):
    a=normalized_name(a); b=normalized_name(b)
    if a==b:return 1.0
    aset=set(a.split()); bset=set(b.split())
    token= len(aset&bset)/max(len(aset|bset),1)
    containment=1.0 if a.replace(' ','') in b.replace(' ','') or b.replace(' ','') in a.replace(' ','') else 0.0
    return max(similarity(a,b), 0.85*token+0.15*containment,0.87*containment)

def normalized_values(series):
    vals=series.dropna().astype(str).str.strip().str.casefold()
    vals=vals[~vals.map(is_blank)]
    vals=vals.drop_duplicates()
    if len(vals)>PROFILE_VALUES: vals=vals.sample(PROFILE_VALUES,random_state=RANDOM_SEED)
    return set(vals.tolist())

value_sets={}
for ds,df in frames.items():
    for j in range(len(df.columns)):
        value_sets[(ds,j)]=normalized_values(df.iloc[:,j])

def overlap(a,b):
    if not a or not b:return 0.0
    # directional containment: partial overlap remains informative when one source is larger
    return len(a&b)/min(len(a),len(b))

pairnames=list(itertools.combinations(frames.keys(),2)) if USE_ALL_PAIRS else SELECTED_PAIRS
for a,b in pairnames:
    if a not in frames or b not in frames:raise ValueError(f'Unknown comparison pair: {a}, {b}')

candidates=[]; summary=[]; missing=[]
for left,right in pairnames:
    la=[meta[(left,i)] for i in range(len(frames[left].columns))]
    rb=[meta[(right,j)] for j in range(len(frames[right].columns))]
    all_pairs=[]
    for i,a in enumerate(la):
      for j,b in enumerate(rb):
        ns=name_sim(a['column'],b['column']); tc=type_compat(a['type'],b['type'])
        ov=overlap(value_sets[(left,i)],value_sets[(right,j)])
        score=round(0.58*ns+0.27*tc+0.15*ov,3)
        all_pairs.append({'Left Dataset':left,'Right Dataset':right,
        'Left Column':a['column'],'Right Column':b['column'],
        'Left Position':i,'Right Position':j,'Left Type':a['type'],'Right Type':b['type'],
        'Name Similarity':round(ns,3),'Type Compatibility':round(tc,3),
        'Value Overlap':round(ov,3),'Overall Score':score,
        'Assessment':'Strong' if score>=0.85 else 'Possible' if score>=0.65 else 'Review',
        'Left Examples':' | '.join(a['examples']), 'Right Examples':' | '.join(b['examples'])})
    pairdf=pd.DataFrame(all_pairs)
    if len(pairdf):
        candidates.extend(pairdf.sort_values('Overall Score',ascending=False).groupby('Left Position',sort=False).head(TOP_MATCHES).to_dict('records'))
    exact_a=set(a['column'] for a in la);exact_b=set(b['column'] for b in rb)
    normalized_a=set(a['normalized'] for a in la);normalized_b=set(b['normalized'] for b in rb)
    for m in la:
        if m['column'] not in exact_b:
            missing.append({'Dataset':left,'Compared To':right,'Column':m['column'],
                            'Suspected Type':m['type'],'Presence':'Only in left (exact name)'})
    for m in rb:
        if m['column'] not in exact_a:
            missing.append({'Dataset':right,'Compared To':left,'Column':m['column'],
                            'Suspected Type':m['type'],'Presence':'Only in right (exact name)'})
    summary.append({'Left Dataset':left,'Right Dataset':right,
      'Left Columns':len(la),'Right Columns':len(rb),'Same Exact Names':len(exact_a&exact_b),
      'Same Normalized Names':len(normalized_a&normalized_b),
      'Only Left (Exact)':len(exact_a-exact_b),'Only Right (Exact)':len(exact_b-exact_a),
      'Candidate Matches Above Threshold':int((pairdf['Overall Score']>=MATCH_THRESHOLD).sum()) if len(pairdf) else 0})

candidate_df=pd.DataFrame(candidates)
if not candidate_df.empty:
    candidate_df=candidate_df[candidate_df['Overall Score']>=MATCH_THRESHOLD].sort_values(
        ['Left Dataset','Right Dataset','Left Position','Overall Score'],ascending=[True,True,True,False])
    best_df=candidate_df.sort_values('Overall Score',ascending=False).drop_duplicates(
        ['Left Dataset','Right Dataset','Left Position']).sort_values(['Left Dataset','Right Dataset','Left Position'])
else:best_df=pd.DataFrame()
summary_df=pd.DataFrame(summary);missing_df=pd.DataFrame(missing)
print('PAIRWISE SUMMARY');display(summary_df)
print('TOP CANDIDATES');display(candidate_df.head(30) if not candidate_df.empty else 'No candidates at this threshold')
print('EXACT-NAME DIFFERENCES');display(missing_df.head(30))

## 4. Export reusable review files
- `column_profiles.csv`: every column, type, completeness, sample values
- `dataset_comparisons.csv`: dataset-pair summary
- `missing_columns.csv`: columns absent by **exact name** (even if a fuzzy equivalent may exist)
- `suggested_matches.csv`: ranked candidate mappings
- `best_matches.csv`: top available suggestion per left column (not exclusive, and only above threshold)
- `all_column_names.csv`: source-to-source column catalog

The best-match file is a **review queue**, not an approved mapping. Filter by `Assessment`, examine examples, and add a manual `Approved` flag if you need production normalization.

In [ ]:
exports = {
 'column_profiles.csv':profiles_df,
 'dataset_comparisons.csv':summary_df,
 'missing_columns.csv':missing_df,
 'suggested_matches.csv':candidate_df,
 'best_matches.csv':best_df,
 'all_column_names.csv':profiles_df[['Dataset','Column','Normalized Name','Suspected Type']] if len(profiles_df) else pd.DataFrame()
}
for filename,frame in exports.items():
    path=OUT_DIR/filename
    frame.to_csv(path,index=False,encoding='utf-8-sig')
    print(f'{filename}: {len(frame)} rows')
print('Output directory:', OUT_DIR.resolve())

## Practical notes

- **False positives:** Numeric phone/ZIP/IDs can look alike; labeled column names help. Common names can be confused with descriptions; treat suspected names as lower confidence.
- **Column-name aliases:** Extend `SYNONYMS` above (e.g. `territory` → `region`, if your business treats them equivalently). Do not synonymize terms that have different business meanings.
- **Comparing many files:** Leave `USE_ALL_PAIRS = True` for a pairwise matrix; set it to `False` and enter `SELECTED_PAIRS` to limit costly comparisons.
- **Large exports:** This loads at most `SAMPLE_ROWS` per dataset, but column-pair comparisons grow with number of columns. Start with 5k–15k rows.
- **Data privacy:** Output includes example values (possibly names, emails, phone numbers). Keep results in approved storage and remove `Examples` fields when sharing.
- **Mapping approval:** Neither type nor name similarity proves that two columns are semantically interchangeable. Dates may be order vs. activation dates, for example.